# Experiment queue of the thesis: session `l4-1`

This notebook runs one session of the job queue in `cloud_runs/` of the repository (see `cloud_runs/README.md`). Before
*Run all*:

1. *Runtime → Change runtime type*: L4 GPU, high RAM.
2. In the first code cell, replace the placeholder in `REPOSITORY_URL` by the address of the repository on GitHub (for
   a private repository, an address that contains an access token), or leave it as it is and put a copy of the
   repository into the store folder as `repository/`.
3. Change `STORE` if the store folder is not `My Drive/aps_thesis`.

Then *Run all*, and allow the access to Google Drive when Colab asks for it. The first code cell mounts Drive and
puts a fresh copy of the repository on the runtime, the second installs the environment (its output also goes to
`logs/environment_l4-1.log` in the store), the third runs the queue until nothing is left for this session, and
the last one releases the runtime so that it stops using compute units.

To move the session to a fresh runtime (a runtime lives at most 24 hours), use *Runtime → Disconnect and delete
runtime* in the notebook that runs it now, then open this notebook and *Run all* again. The jobs continue from their
checkpoints in the store. If the old runtime is still alive, the new one waits up to 20 minutes and then gives up, so
that nothing runs twice.

The state of the queue can be read from any notebook or terminal that sees the store:
`python cloud_runs/queue_runner.py --store <store folder> --status`.

In [ ]:
SESSION = "l4-1"   # a100-1, a100-2, a100-3, l4-1, l4-2, cpu-1 (see cloud_runs/colab/colab_bootstrap.py)
STORE = "/content/drive/MyDrive/aps_thesis"   # the shared store folder in Google Drive
# FILL IN the address of the repository on GitHub, or leave the placeholder and copy the repository to STORE/repository
REPOSITORY_URL = "https://github.com/<account>/scania-aps-synthetic-failure-data.git"
REPOSITORY_DIR = "/content/scania-aps-synthetic-failure-data"

import os, shutil, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
os.chdir("/content")
shutil.rmtree(REPOSITORY_DIR, ignore_errors=True)   # a fresh copy of the repository for every Run all
if "<account>" not in REPOSITORY_URL:
    subprocess.run(["git", "clone", "--quiet", REPOSITORY_URL, REPOSITORY_DIR], check=True)
elif os.path.isdir(f"{STORE}/repository"):
    shutil.copytree(f"{STORE}/repository", REPOSITORY_DIR)
else:
    raise RuntimeError(f"fill in REPOSITORY_URL, or copy the repository to {STORE}/repository")
os.chdir(REPOSITORY_DIR)                             # the following cells run in the repository
sys.path.insert(0, f"{REPOSITORY_DIR}/cloud_runs/colab")
import colab_bootstrap
settings = colab_bootstrap.prepare_session(SESSION, STORE)
LANES, SLOTS, IDLE_EXIT_MINUTES = settings.lanes, settings.slots, settings.idle_exit_minutes

In [ ]:
# packages, data, the external repositories and the exact inputs (full output: logs/environment_<session>.log in the store)
!bash cloud_runs/setup_environment.sh 2>&1 | tail -60

In [ ]:
# the queue: runs this session's jobs until nothing is left for IDLE_EXIT_MINUTES or the session is drained
!python cloud_runs/queue_runner.py --store "{STORE}" --session "{SESSION}" --lanes "{LANES}" --slots {SLOTS} --idle-exit-minutes {IDLE_EXIT_MINUTES}

In [ ]:
# hands the runtime back to Colab once the queue has finished for this session
colab_bootstrap.release_runtime_if_queue_finished(SESSION, STORE, REPOSITORY_DIR)